Loading the deconvoluted spectra using a dataloader

In [4]:
import torch 
from torch.utils.data import Dataset, DataLoader
from pyteomics import mgf
import numpy as np
import logging
import torch.nn as nn
import torch
import torch.nn.functional as F
from torch import optim

In [3]:
def get_bin_index(self, mz, min_mz, bin_size):
    relative_mz = mz - min_mz
    return max(0, int(np.floor(relative_mz / bin_size)))

def bin_spectrum(self, mz_array, intensity_array, max_mz=2500, min_mz=50.5, bin_size=1.0005079):
    """
    bin spectrum and this algorithm reference from 'https://github.com/dhmay/param-medic/blob/master/parammedic/binning.pyx'
    :param mz_array:
    :param intensity_array:
    :param max_mz:
    :param min_mz:
    :param bin_size:
    :return:
    """
    # key = mz_array.__str__()
    # if key in spectrum_dict.keys():  # use cache just take 4s
    #     # if False: use the old one may take 7s for 50
    #     return spectrum_dict[key]
    # else:
    nbins = int(float(max_mz - min_mz) / float(bin_size)) + 1
    results = np.zeros(nbiEns)

    for index in range(len(mz_array)):
        mz = mz_array[index]
        intensity = intensity_array[index]
        intensity = np.math.sqrt(intensity)
        if mz < min_mz or mz > max_mz:
            continue
        bin_index = self.get_bin_index(mz, min_mz, bin_size)

        if bin_index < 0 or bin_index > nbins - 1:
            continue
        if results[bin_index] == 0:
            results[bin_index] = intensity
        else:
            results[bin_index] += intensity

    intensity_sum = results.sum()

    if intensity_sum > 0:
        results /= intensity_sum
        # spectrum_dict[key] = results
    else:
        logging.debug('zero intensity found')
    return results


In [4]:
class SpectrumDataset(Dataset):
    def __init__(self,mgf_file_path, max_peaks=200):
        self.mgf_file_path = mgf_file_path
        self.max_peaks = max_peaks
        self.spectra = list(mgf.read(mgf_file_path))
        
    def __len__(self):
        return len(self.spectra)

    def __getitem__(self, index):
        spectrum = self.spectra[index]
        mz_values = spectrum.get("m/z array", np.array([]))
        intensity_values = spectrum.get("intensity array", np.array([]))
        
        # normalize intensities
        intensity_values = intensity_values / np.max(intensity_values) if intensity_values.size else intensity_values

        return torch.tensor(np.stack(mz_values,intensity_values), axis=1)

    

In [ ]:
mgf_file_path = "./spectraList.mgf"
spectrum = list(mgf.read(mgf_file_path))    

In [21]:
list(spectrum)[1].get("m/z array")

array([ 216.188,  217.13 ,  223.079, ..., 1533.15 , 1542.602, 1545.948])

In [ ]:
mgf_file_path = "./spectraList.mgf"


dataset = SpectrumDataset(mgf_file_path)

Training_data_loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=1 )


In [8]:
class SiamesNetwork(nn.Module):
    def __init__(self):
        super(SiamesNetwork, self).__init__()
        
        self.fc1 = nn.Sequential(
            nn.Linear(34,32),
            nn.SELU(),
            nn.Linear(32,5),
            nn.SELU()            
        )
        self.cnn11 = nn.Sequential(
            nn.Conv1d(1,30,3),
            nn.SELU(),
            nn.MaxPool1d(2),
            nn.SELU()
        )
        self.cnn21 = nn.Sequential(
            nn.Conv1d(1,30,3),
            nn.SELU(),
            nn.MaxPool1d(2),
            nn.SELU(),
            nn.Conv1d(30,30,3),
            nn.MaxPool1d(2),
            nn.SELU()
        )
        self.fc2 =nn.Linear(25775,32)
    
    def forward_once(self, x):
        # Get feature embeddings
        output = self.fc1(x)
        output = self.cnn11(x)
        output = self.cnn21(x)
        output = self.fc2(x)
        
        # Compute Euclidean distance
        return output
    
    def forward(self, x1, x2):
        output1 = self.forward_once(x1)
        output2 = self.forward_once(x2)
        return output1, output2

In [12]:
# Define the Contrastive Loss Function
class ContrastiveLoss(torch.nn.Module):
    def __init__(self, margin=2.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, output1, output2, label):
      # Calculate the euclidian distance and calculate the contrastive loss
      euclidean_distance = F.pairwise_distance(output1, output2, keepdim = True)

      loss_contrastive = torch.mean((1-label) * torch.pow(euclidean_distance, 2) +
                                    (label) * torch.pow(torch.clamp(self.margin - euclidean_distance, min=0.0), 2))


      return loss_contrastive

In [15]:
net =SiamesNetwork()
lossFunc = ContrastiveLoss()
optimzer = optim.Adam(net.parameters(), lr = 0.0005)

In [ ]:
epoches = 10

for epoch in range(0, epoches):
    for i, data in enumerate(Training_data_loader,0):
        print()